# Imports

In [0]:
import pickle
import re
import subprocess
from datetime import datetime
from pathlib import Path
from mlflow import MlflowClient
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Parâmetros do notebook

In [0]:
# dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")

In [0]:
dbutils.widgets.text("id_projeto", "endometriose", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

print(f"environment_tbl: {environment_tbl}")

In [0]:
tbl_entrada = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_entrada"
tbl_pre_processamento = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_pre_processamento"
tbl_saida = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_saida"

In [0]:
def print_params():
    print(f"Data Execução Modelo : {data_execucao_modelo}")
    print(f"id_projeto           : {id_projeto}")
    
        
    print(f"environment          : {environment}")
    print(f"environment_tbl      : {environment_tbl}")
    print(f"tbl_entrada          : {tbl_entrada}")
    print(f"tbl_pre_processamento: {tbl_pre_processamento}")
    
    print(f"tbl_saida            : {tbl_saida}")
    

print_params()

# Recupera artefatos

## Production

## Staging

# Data

In [0]:
query = f"""
    SELECT
        dataExecucaoModelo,
        idPredicao,
        idExame,
        idPaciente,
        descricaoProcedimento,
        laudoOriginal,
        laudoResumido,
        listaSintomas,
        listaAchados,
        classificacao,
        date(dataExame) as dataExame
    from ia.{tbl_pre_processamento}
    where dataExecucaoModelo = '{data_execucao_modelo}'
"""

df_spark = spark.sql(query)

if df_spark.rdd.isEmpty():
  dbutils.notebook.exit("Não há dados para serem processados! Execução cancelada!")

In [0]:
from pyspark.sql.functions import col, size

count_non_empty = df_spark.filter((size(col("listaSintomas")) > 0) | (size(col("listaAchados")) > 0)).count()

print(count_non_empty)

# Predição

In [0]:
# Lista com os termos de interesse
sintomas_endometriose = [
    "dor pelvica cronica", "dismenorreia", "dispareunia", "pesquisa de infertilidade",
    "infertilidade", "urgencia miccional", "disuria", "hematuria"
]

achados_endometriose = [
    "endometrioma", "dilatacao unilateral do ureter", "espessamento do ligamento uterossacro",
    "ligamento uterossacros", "ligamentos uterossacros", "espessamento dos ligamentos uterossacros",
    "espessamento significativo dos ligamentos uterossacros", "espessamento irregular do ligamento uterossacro",
    "espessamento na topografia dos ligamentos uterossacros", "espessando os ligamentos uterossacros",
    "deflexinal do ligamento uterossacro", "espessamento, hipossinal e irregularidade da porcao intraperitoneal dos ligamentos uterossacros",
    "espessamento ligamento uterossacro", "endometriotica", "endometriotico", "endometriose", "endometriomas",
    "lesoes peritoneais", "lesoes tubarias", "lesoes ovarianas", "lesoes infiltrativas"
]

negacoes = [
    "nao ha", "sem", "ausencia de", "negado", "não existe", "sem evidencia de", "nao ha sinais", "nao identifico sinais",
    "sem nitidas alteracoes", "sem alteracoes", "preservados", "preservado", "sem nitidas alteracoes relevantes para",
    "espessamento regular", "nao ha sinais de", "nao foram caracterizadas lesoes", "sem falha de enchimento",
    "nao ha lesao", "estruturas estao livres", "nao ha cistos hematicos", "sem alteracoes evidentes",
    "nao ha cistos hematicos e/ou", ": livres", "aparentemente livre", "nao ha nitidos sinais de",
    "nao se observam sinais", "com espessura e sinal conservados", "redondos com espessura mantida",
    "nao se caracterizam sinais de", "nao ha cistos com conteudo", "que isoladamente e inespecifico", "espessura normal",
    "nao ha imagens cistos hematicos e / ou", "nao ha imagens cistos hematicos e/ou", "nao ha sinais evidentes de",
    "nao foram caracterizadas imagens compativeis com", "exame sem sinais de", ": nao individualizados",
    "espessamento liso dos", "pois nao e possivel descartar totalmente", "achado inespecifico", "sem alteracoes evidentes",
    "minimo espessamento do ligamento uterossacro direito, inespecifico", "sem lesoes suspeitas para", "hipossinal dos",
    "hipointensos da origem ate o segmento lateral", "nao ha sangramento ou", "nao ha lesao",
    "pois nao e possivel descartar totalmente", "espessamento regular dos segmentos", "nao se observam cistos hematicos",
    "bexiga urinaria com morfologia conservada", " nao ha imagens de cistos hematicos", "sem alteracoes evidentes",
    "canal vaginal: sem alteracoes evidentes", "septo retovaginal: sem alteracoes evidentes",
    "apenas na dependencia de correlacao", "nao se observam francos implantes", "ligamentar apenas se houver correlacao com exame fisico",
    "nao foram evidenciados sinais sugestivos de", "nao sao observados cistos hematicos e / ou", "nao apresentam espessamentos significativos",
    "ausencia de lesoes suspeitas para", "nao se identificou", "sem focos hematicos associados", "sem focos hematicos ou processo aderencial",
    "ao ha cistos com conteudo hematico ou", "nao ha espessamento significativo / nao ha sinais de",
    "valorizar para endometriose ligamentar apenas se houver correlacao com o exame fisico", "superficial nem",
    "nao ha sinais de adenomiose uterina, sangramento ou", "caso exista a suspeita clinica de", "sem falha de enchimento, sangramento ou",
    "nao se observa", "e septo retovaginal preservados", "nao ha nodulos, cistos com conteudo hematico ou",
    "nao ha formacoes expansivas anexiais, cistos com conteudo hematico ou", "paciente decorrentes de urgencia miccional",
    "negativos nao excluem a possibilidade de endometriose", "de espessura e sinal conservados",
    "nao se observam alteracoes definitivas para", "nao ha espessamento ou sangramento na projecao dos",
    "compativeis com miomas\nnao foram caracterizados sinais de", "nao se observam francos implantes",
    "associadamente nao percebo mais o aspecto nodular do ligamento redondo a esquerda ou mesmo irregularidade dos",
    "nao ha cistos hematicos e / ou"
]

confirmacoes = [
    "sugestiva de", "sugestivo de", "indicativo de", "confirmado", "detectado", "sugerindo",
    "suspeita para", "que pode estar associado a", "pois nao e possivel afastar", "associado a espessamentos do",
    "associado a espessamento do", "hipointenso", "discretamente espessado", "espessamento insercional",
    "levemente irregulares", "hipointensos e t2", "discreto espessamento fibroso", "lesao endometriotica",
    "irregularidade e hipossinal", "compativel com lesao", "espessamento laminar fibrotico retrocervical, associado a",
    "discreto espessamento proximal com baixo sinal dos"
]

In [0]:
def contexto_negacao(texto, palavra):
    # Define o padrão para encontrar a palavra-alvo próxima a qualquer termo de negação
    padrao_negacao = r'\b(?:' + '|'.join(re.escape(neg) for neg in negacoes) + r')\b.{0,70}?\b' + re.escape(palavra) + r'\b|\b' + re.escape(palavra) + r'\b.{0,70}?\b(?:' + '|'.join(re.escape(neg) for neg in negacoes) + r')\b'
    return bool(re.search(padrao_negacao, texto, re.IGNORECASE))


def contexto_confirmacao(texto, palavra):
    # Define o padrão para encontrar a palavra-alvo próxima a qualquer termo de confirmação
    padrao_confirmacao = r'\b(?:' + '|'.join(re.escape(conf) for conf in confirmacoes) + r')\b.{0,70}?\b' + re.escape(palavra) + r'\b|\b' + re.escape(palavra) + r'\b.{0,70}?\b(?:' + '|'.join(re.escape(conf) for conf in confirmacoes) + r')\b'
    return bool(re.search(padrao_confirmacao, texto, re.IGNORECASE))


def capturar_sentenca(texto, palavra):
    sentencas = re.split(r'[.!?]', texto)
    sentencas_relevantes = []
    
    for sentenca in sentencas:
        sentenca_limpa = sentenca.strip()
        palavras = sentenca_limpa.split()
        
        if (palavra.lower() in sentenca_limpa.lower() and
            'indicacao:' not in sentenca_limpa.lower() and
            'indicacao do exame:' not in sentenca_limpa.lower() and
            'indicacao clinica:' not in sentenca_limpa.lower() and
            'status pos' not in sentenca_limpa.lower() and
            'historico' not in sentenca_limpa.lower() and
            'obs:' not in sentenca_limpa.lower() and
            'paciente relata urgencia miccional' not in sentenca_limpa.lower() and
            'pode nao ser identificada ao metodo' not in sentenca_limpa.lower() and
            'aspecto fibrocicatricial pos-cirurgico' not in sentenca_limpa.lower() and
            'pesquisa:' not in sentenca_limpa.lower() and
            'pesquisa' not in sentenca_limpa.lower() and
            'relato' not in sentenca_limpa.lower() and
            'pos-cirurgica' not in sentenca_limpa.lower() and
            'pos-cirurgicas' not in sentenca_limpa.lower() and
            'dados clinicos:' not in sentenca_limpa.lower() and
            'nota:' not in sentenca_limpa.lower() and
            'exame realizado em' not in sentenca_limpa.lower() and
            'mapeamento:' not in sentenca_limpa.lower() and
            'analise comparativa' not in sentenca_limpa.lower() and
            'investigacao' not in sentenca_limpa.lower() and 
            'informacao clinica:' not in sentenca_limpa.lower() and
            'informacao disponivel:' not in sentenca_limpa.lower() and
            'descricao:' not in sentenca_limpa.lower() and
            'comparacao:' not in sentenca_limpa.lower() and
            'bexiga sob baixa replecao, com conteudo homogeneo' not in sentenca_limpa.lower() and
            'paciente refira urgencia miccional' not in sentenca_limpa.lower() and
            'tecnica:' not in sentenca_limpa.lower() and
            len(palavras) > 1):  # Ignorar sentenças de uma única palavra
            sentencas_relevantes.append(sentenca_limpa)
    
    return sentencas_relevantes


def classificar_laudo(laudo, sintomas, achados):
    sentencas_sintomas = [capturar_sentenca(laudo, sintoma) for sintoma in sintomas]
    sentencas_achados = [capturar_sentenca(laudo, achado) for achado in achados]

    sentencas_sintomas = [sentenca for sublist in sentencas_sintomas for sentenca in sublist if sublist]
    sentencas_achados = [sentenca for sublist in sentencas_achados for sentenca in sublist if sublist]

    positivo_sintoma = any(not contexto_negacao(sentenca, sintoma) for sintoma in sintomas for sentenca in sentencas_sintomas if sintoma in sentenca)
    positivo_achado = any(not contexto_negacao(sentenca, achado) for achado in achados for sentenca in sentencas_achados if achado in sentenca)
    confirmado_endometriose = any(contexto_confirmacao(sentenca, achado) for achado in achados for sentenca in sentencas_achados if achado in sentenca)

    return 'Positivo' if (positivo_sintoma or positivo_achado or confirmado_endometriose) else 'Negativo'

In [0]:
def classifica(texto):
    if texto is None:
        return ""

    str_mod = classificar_laudo(texto, sintomas_endometriose, achados_endometriose)
    
    return str_mod

# Registrando a função clean_laudo como UDF no Spark
classifica_udf = udf(classifica, StringType())
spark.udf.register("classifica_udf", classifica_udf)

In [0]:
df_modificado = df_spark.withColumn("classificacaoEndometriose", classifica_udf(df_spark["laudoResumido"]))

In [0]:
df_filtered = df_modificado.filter((size(df_modificado["listaSintomas"]) >= 1) | (size(df_modificado["listaAchados"]) >= 1))
# Exibir o novo DataFrame filtrado
df_filtered.display()

# Salva os dados na tabela de saída

In [0]:
tbl_saida

In [0]:
table_exists = spark.catalog.tableExists(f"ia.{tbl_saida}")

if table_exists:
    spark.sql(f"""
        delete from ia.{tbl_saida}
        where dataExecucaoModelo = '{data_execucao_modelo}'
    """).display()

In [0]:
df_filtered.write.mode("append").option("mergeSchema", "true").partitionBy("dataExecucaoModelo").saveAsTable(f"ia.{tbl_saida}")

In [0]:
spark.sql(f"vacuum ia.{tbl_saida}").display()
spark.sql(f"optimize ia.{tbl_saida}").display()
spark.sql(f"analyze table ia.{tbl_saida} compute statistics").display()

In [0]:
spark.sql(f"""
    select
        dataExecucaoModelo,
        count(1) as total,
        count_if(classificacaoEndometriose = "Positivo") as Positivo,
        count_if(classificacaoEndometriose = "Negativo") as Negativo
    from ia.{tbl_saida}
    group by all
    order by dataExecucaoModelo desc
""").display()

In [0]:
spark.sql(f"""
    select
        idExame,
        count(1) as qtd
    from ia.{tbl_saida}
    group by all
    having qtd > 1
""").display()

In [0]:
dbutils.notebook.exit("Fim do processamento!")

# Fim do processamento!